# Four-class sonar segmentation: patch experiments

Run the configuration and dataset setup cells first, then run each experiment independently.
K-means, CNN at two patch sizes, and U-Net at two patch sizes each report loss, validation
mean/per-class IoU, and **four patch rows** (raw / class-colored ground truth / prediction).
If fewer than four validation patches exist, all available patches are shown.

Expected input (matching filename stems, potentially different extensions):
```
data/
  image/scan_01.png
  mask/0/scan_01.png
  mask/1/scan_01.png
  mask/2/scan_01.png
  mask/3/scan_01.png
```
Exactly four numeric mask folders are required; their sorted numbers define class IDs 0–3
(e.g. folders `0, 1, 7, 10` map to IDs `0, 1, 2, 3`). Masks may be single-channel or RGB/RGBA PNGs:
black means absent; white (or any nonzero color channel) means present. Alpha is ignored. Masks must match the source image dimensions.
Overlapping masks raise an error. Inputs are 8-bit grayscale (RGB is converted to grayscale).
No example data or trained weights are bundled.

Install `requirements.txt` in your kernel environment before running. For CPU-only machines,
install `torch==2.7.1 torchvision==0.22.1` from `https://download.pytorch.org/whl/cpu` first.
The existing cloud `.venv` is prepared. Select that environment's Python kernel.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

# Open this notebook from the repository root (or its notebooks/ directory).
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "experiments.py").exists() and (REPO_ROOT.parent / "experiments.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "experiments.py").exists():
    raise RuntimeError("Start Jupyter in the side-scan-sonar-segmentation repository")
sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = REPO_ROOT / "data"  # Change to the directory containing image/ and mask/.
IMAGE_FOLDER = "image"
NUM_CLASSES = 4
PATCH_SIZES = (128, 256)
EPOCHS = 5
BATCH_SIZE = 4  # Lower this for large patches / limited GPU memory.
LEARNING_RATE = 1e-3
VALIDATION_FRACTION = 0.2
SEED = 42
# Default rejects uncovered pixels. Set to 0..3 ONLY if uncovered pixels mean that class.
BACKGROUND_CLASS = None
MAX_KMEANS_PIXELS = 200_000
KMEANS_TEMPERATURE = 0.02
DEVICE = None  # Auto-select CUDA if available; set "cpu" to force CPU.


In [ ]:
import torch
from dataset.PatchDataset import discover_samples, split_samples, PatchDataset
from models.BasicKmeans import BasicKMeans
from models.BasicCNN import BasicCNN
from models.BasicUnet import BasicUNet
from experiments import run_experiment

samples, class_names = discover_samples(DATA_ROOT, IMAGE_FOLDER)
assert NUM_CLASSES == len(class_names) == 4
train_samples, validation_samples = split_samples(samples, VALIDATION_FRACTION, SEED)
print(f"Source images: train={len(train_samples)}, validation={len(validation_samples)}")
print("Class ID -> mask folder:", dict(enumerate(class_names)))
print("Device:", DEVICE or ("cuda" if torch.cuda.is_available() else "cpu"))
for patch_size in PATCH_SIZES:
    print(f"Patch {patch_size}: train={len(PatchDataset(train_samples, patch_size, BACKGROUND_CLASS))}, "
          f"validation={len(PatchDataset(validation_samples, patch_size, BACKGROUND_CLASS))}")

# Every experiment starts from a fresh model with the same source-image split and seed.
common = dict(train_samples=train_samples, validation_samples=validation_samples,
              class_names=class_names, epochs=EPOCHS, batch_size=BATCH_SIZE,
              learning_rate=LEARNING_RATE, seed=SEED, background_class=BACKGROUND_CLASS,
              device=DEVICE)
results = {}


## K-means baseline
Four intensity clusters are fitted on at most `MAX_KMEANS_PIXELS` sampled **training** pixels.
A one-to-one cluster/class assignment is selected using training masks only; validation labels
never calibrate the mapping. This is a label-calibrated clustering baseline.

Reported inertia is K-means' training objective. The reported cross-entropy loss uses
softmax of negative squared intensity-to-centroid distances divided by `KMEANS_TEMPERATURE`;
it is a diagnostic probability model, not a learned neural loss or a calibrated confidence.
K-means does not use spatial context; its patch size controls tiling and sampling.

In [ ]:
results["kmeans"] = run_experiment(
    "kmeans", PATCH_SIZES[0], max_kmeans_pixels=MAX_KMEANS_PIXELS,
    kmeans_temperature=KMEANS_TEMPERATURE, **common)


## CNN: smaller patches
Fresh fully convolutional model; one grayscale input channel, four output logits per pixel.

In [ ]:
results["cnn_small"] = run_experiment("cnn", PATCH_SIZES[0], **common)


## CNN: larger patches
Same architecture and training settings, fresh initialization, larger spatial context.

In [ ]:
results["cnn_large"] = run_experiment("cnn", PATCH_SIZES[1], **common)


## U-Net: smaller patches
Two encoder/decoder stages with skip connections; one input channel and four output logits.

In [ ]:
results["unet_small"] = run_experiment("unet", PATCH_SIZES[0], **common)


## U-Net: larger patches

In [ ]:
results["unet_large"] = run_experiment("unet", PATCH_SIZES[1], **common)


## Compare completed experiments
All metrics use the held-out source images. Border tiles are padded; padded pixels are
excluded from loss, IoU, and plots. IoU is computed from the aggregate validation confusion
matrix, not averaged across batches. Classes absent in both truth and prediction are NaN
and excluded from mean IoU. Colors stay fixed across experiments.

A fixed number of epochs gives different optimizer-step counts at different patch sizes;
this is an initial patch-size comparison, not a controlled compute-budget benchmark.
There is no independent test split: use separate test images for final performance reporting.
Real training quality must be assessed on your own sonar data.

In [ ]:
print(f"{'Experiment':<18} {'Patch':>6} {'Val loss':>12} {'Mean IoU':>12}")
for name, result in results.items():
    print(f"{name:<18} {result['patch_size']:>6} "
          f"{result['validation_loss']:>12.6f} {result['mean_iou']:>12.4f}")
